# Statistical testing

Several grouped bar charts in the EDA look like they show real
relationships, treatment scaling with severity of work interference,
for both remote workers and tech company employees. Chi-square tests
check whether that's actually true.

In [1]:
import pandas as pd
from scipy import stats

df = pd.read_pickle('cleaned_mhealth.pkl')
df.shape

(992, 25)

In [2]:
def chi_square_test(column, target = 'work_interfere'):
    ct = pd.crosstab(df[column], df[target])
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    return chi2, p

for col in ['remote_work', 'tech_company', 'no_employees', 'family_history', 'Gender']:
    chi2, p = chi_square_test(col)
    print(f'{col:15s} vs work_interfere: chi2 = {chi2:7.2f}, p = {p:.4f}')

remote_work     vs work_interfere: chi2 =    2.92, p = 0.4040
tech_company    vs work_interfere: chi2 =    3.30, p = 0.3470
no_employees    vs work_interfere: chi2 =   23.19, p = 0.0802
family_history  vs work_interfere: chi2 =   83.46, p = 0.0000
Gender          vs work_interfere: chi2 =   18.59, p = 0.0049


Two of the visual reads don't survive formal testing. `remote_work`
(p = 0.404) and `tech_company` (p = 0.347) show **no** significant
association with `work_interfere`, despite both bar charts looking like
a proportional relationship. `no_employees` (company size) sits at
p = 0.080, not significant at the conventional 0.05 cutoff either, a
weaker signal than the chart alone might suggest. `family_history`
(p < 0.0001) and `Gender` (p = 0.005) are the two factors that actually
hold up.

In [3]:
chi2, p = chi_square_test('family_history', target = 'treatment')
print(f'family_history vs treatment: chi2 = {chi2:.2f}, p = {p:.2e}')

pd.crosstab(df.family_history, df.treatment, normalize = 'index').round(3)

family_history vs treatment: chi2 = 104.91, p = 1.28e-24


treatment,No,Yes
family_history,,
No,0.507,0.493
Yes,0.191,0.809


Having a family history of mental illness is very strongly associated with seeking treatment, respondents with a family history seek treatment at a much higher rate. This one holds up clearly, both visually and formally.

In [4]:
import json, os
os.makedirs('outputs', exist_ok = True)
results = {}
for col in ['remote_work', 'tech_company', 'no_employees', 'family_history', 'Gender']:
    chi2, p = chi_square_test(col)
    results[col] = {'chi2': float(chi2), 'p': float(p)}
with open('outputs/statistical_tests.json', 'w') as f:
    json.dump(results, f, indent = 2)